# Conversion of image intensity to protein concentration

Brightfield intensity and total protein (measured via BCA assay) were observed possessing a direct relationship on a log-log scale pointing to a power law relationship. Conversion from 16-bit intensity to $\mu$g/mL aims to accomplish bulk protein concentration agreement around the endpoint (t = 20 hour) measurement.

- **Intercept of power law relationship is set to regressed value arrived by previous calibration experiment**

In [3]:
import numpy as np
import pandas as pd
import glob
import os
from scipy.optimize import minimize, Bounds, fmin
from scipy.integrate import simpson
from scipy import stats

class TimeLapseFile():
    @staticmethod
    def nan_remove(arr): #Input: np.array
        arr2=np.where(np.isnan(arr))
        filt_arr=np.delete(arr,arr2)
        return filt_arr
    
    @staticmethod
    def nan_adding(arr_base,arr1): #Input: np.array
        resize_arr1=list(np.copy(arr1))
        if len(arr_base) != len(resize_arr1):
            for j in range(int(len(arr_base)-len(resize_arr1))):
                resize_arr1.append(np.nan)
        return np.array(resize_arr1)
    
    def __init__(self,xlsx_files,time_offset):
        self.ws_set=[]
        self.sheets_set=[]
        self.r_pos=[]
        self.t_idx=[]
        self.t_arr=[]
        self.sample_data=[]
        self.bkgrd_data=[]
        self.t_displace=time_offset
        for j in range(len(xlsx_files)):
            self.ws_set.append(xlsx_files[j])
            self.sheets_set.append(pd.ExcelFile(xlsx_files[j]).sheet_names)
            
    def timeseries_split(self):
        for j in range(len(self.ws_set)):
            group_rad=[]
            group_sample=[]
            group_bkgrd=[]
            for k in range(len(self.sheets_set[j])):
                df_ix=pd.read_excel(self.ws_set[j],sheet_name=self.sheets_set[j][k])
                group_rad.append(df_ix[df_ix.columns[0]].sub(0.1))
                group_bkgrd.append(df_ix[df_ix.columns[1]])
                group_sample.append(df_ix[df_ix.columns[2:]])
            self.r_pos.append(group_rad)
            self.sample_data.append(group_sample)
            self.bkgrd_data.append(group_bkgrd)
        self.t_idx=np.linspace(0,group_sample[0].shape[1],group_sample[0].shape[1]+1)
        self.t_arr=np.array([self.t_idx[x] + self.t_displace for x in range(len(self.t_idx))])
        return print(f"Start point of position arrays subtracted 0.1 cm")

class EndPointFile(TimeLapseFile):
    def __init__(self, xlsx_files,time_offset):
        super().__init__(xlsx_files,time_offset)
    
    def endpoint_split(self):
        group_rad=[]
        group_sample=[]
        group_bkgrd=[]
        for j in range(len(self.ws_set)):
            for k in range(len(self.sheets_set[j])):
                df_ix=pd.read_excel(self.ws_set[j],sheet_name=self.sheets_set[j][k])
                if 'Sample' in self.sheets_set[j][k]:
                    group_rad.append(df_ix[df_ix.columns[0]].sub(0.1))
                    group_sample.append(df_ix[df_ix.columns[1:]])
                else:
                    group_bkgrd.append(df_ix[df_ix.columns[1:]])
        self.r_pos=group_rad
        self.sample_data=group_sample
        self.bkgrd_data=group_bkgrd
        return print(f"Start point of position arrays subtracted 0.1 cm")

def radinteg_1D(r, c_i):
    return 2.0*np.pi*simpson(r*c_i,r)
    
def avgarea_cyl1D(r1,r2):
    #annulus when r1 != 0
    return np.pi*(r2**2 - r1**2)

def spatial_variance(x_i, spatial_avg):
    return sum([(x_i[g]-spatial_avg)**2 for g in range(len(x_i))])/(len(x_i)-1)

In [4]:
#Function instance endpoint
def optimize_brightfld_prot(img_profile, img_bkgrd, rpos, biom_target, param_pass):
    #param_pass (list): power law [exponent, slope, intercept], intercept set the regression value
    init_guess=param_pass[:2]
    cons_func=param_pass[2]
    
    def power_limfunc(param_arr, x1=img_profile, x2=img_bkgrd, dr=rpos, target=biom_target, set_param=cons_func):
        a, b = param_arr
        y1=np.zeros(len(x1))
        sat_diff= set_param + b*(65535**a) # selection of 'a' and 'b' not to unbalance a near-zero protein concentration at saturation
        for i in range(len(x1)):
            pt1 = b*(x1[i]**a)
            pt2 = b*(x2[i]**a)
            if pt1 - pt2 > 0:
                y1[i] = pt1 - pt2
            elif pt1 - pt2 < 0:
                y1[i] = 0.0
        biom_avg=radinteg_1D(dr,y1)/avgarea_cyl1D(0,dr[-1])
        return (abs(biom_avg-target) + abs(sat_diff))
    
    res_opt=minimize(power_limfunc,init_guess,method='Nelder-Mead',options={'fatol':1e-8})
    prof_check=np.zeros(len(img_profile))
    a1, b1 = res_opt.x
    for j in range(len(img_profile)):
        pt_sample=b1*img_profile[j]**a1
        pt_bkgrd=b1*img_bkgrd[j]**a1
        if pt_sample - pt_bkgrd > 0:
            prof_check[j] = pt_sample - pt_bkgrd
        elif pt_sample - pt_bkgrd < 0:
            prof_check[j] = 0.0
            
    biom_check=radinteg_1D(rpos,prof_check)/avgarea_cyl1D(0,rpos[-1])
    return res_opt.x, abs(biom_check-biom_target)

def convert_brightfld_prot(img_profile, img_bkgrd, power_param):
    a_val, b_val = power_param
    conv_profile=np.zeros(len(img_profile))
    for j in range(len(img_profile)):
        pt_sample=b_val*img_profile[j]**a_val
        pt_bkgrd=b_val*img_bkgrd[j]**a_val
        if pt_sample - pt_bkgrd > 0:
            conv_profile[j] = pt_sample - pt_bkgrd
        elif pt_sample - pt_bkgrd < 0:
            conv_profile[j] = 0.0
    return conv_profile

# Time-series image data

In [ ]:
os.chdir("path/to/Swimplate_Data/Processed-intensity-profiles/")
str_f=['del metH']#"WT_025ag","WT_028ag","del metH"]
tlapse_set=[glob.glob(os.getcwd()+f"/SwmPlt-intensity_tlapse*{x}*.xlsx")[0] for x in str_f]
profile_tlapse=TimeLapseFile(tlapse_set,time_offset=6)
profile_tlapse.timeseries_split()

#Biomass Targets
biom_targ=[]
df_biom=pd.read_excel("TotalProtein_BiomTarget.xlsx", sheet_name='Timelapse')
for j in range(len(str_f)):
    biom_targ.append(df_biom[df_biom["Group"]==str_f[j]].loc[:,df_biom.columns[1:]].to_numpy()[0])

Start point of position arrays subtracted 0.1 cm


In [12]:
prof_bkgrdsub=[]
res_powerparam=[]
for j in range(len(profile_tlapse.ws_set)):
    val_avgbiom=[{} for x in range(len(profile_tlapse.sheets_set[j]))]
    power_paramopt=pd.DataFrame(columns=["Exponent", "Slope"])
    
    #evaluate the longest radial position array
    rad_shape=[]
    rpos_hold=[]
    for x in range(len(profile_tlapse.ws_set)):
        for v in range(len(profile_tlapse.sheets_set[x])):
            rad_shape.append(profile_tlapse.sample_data[x][v].shape[0])
            r_xv=profile_tlapse.r_pos[x][v].to_numpy()
            discr_xv=np.zeros(len(r_xv)//42)
            for idr in range(len(discr_xv)):
                discr_xv[idr] = np.mean(r_xv[42*idr:42*(idr+1)])
            rpos_hold.append(discr_xv)
    ind_radmax=rad_shape.index(max(rad_shape))
    
    #---Spatial averaging---#
    for k in range(len(profile_tlapse.sheets_set[j])):
        r_ix=profile_tlapse.r_pos[j][k].to_numpy()
        bkgrd_ix=profile_tlapse.bkgrd_data[j][k]
        sample_idt=profile_tlapse.sample_data[j][k]
        discr_r=np.zeros(len(r_ix)//42) #floor division towards spatial averaging of ~400 um
        discr_bkgrd=np.zeros(len(r_ix)//42)
        discr_samp_avg=np.zeros((len(r_ix)//42,sample_idt.shape[1]))
        discr_samp_std=np.zeros((len(r_ix)//42,sample_idt.shape[1]))
        for idx in range(len(discr_r)):
            dr=r_ix[42*idx:42*(idx+1)]
            discr_r[idx]=np.mean(dr)
            discr_bkgrd[idx]=radinteg_1D(dr,bkgrd_ix[42*idx:42*(idx+1)])/avgarea_cyl1D(dr[0],dr[-1])
            for n in range(sample_idt.shape[1]):
                gry_dr=sample_idt[sample_idt.columns[n]].to_numpy()[42*idx:42*(idx+1)]
                avg_gry=radinteg_1D(dr,gry_dr)/avgarea_cyl1D(dr[0],dr[-1])
                discr_samp_avg[idx, n] = avg_gry

        #---Power Law optimization---#
        val_avgbiom[k]["Position (cm)"]=pd.Series(rpos_hold[ind_radmax])
        biom_ix=biom_targ[j][k]
        param_opt, fit_error = optimize_brightfld_prot(discr_samp_avg[:,-1], discr_bkgrd, discr_r, biom_ix, [-1.1, 400000., -1126.]) # set intercept: -1126.
        print(f"Minimization complete with experiment disagreement of: {fit_error}")
        power_paramopt.loc[len(power_paramopt)]=param_opt
        for idy in range(sample_idt.shape[1]):
            pre_conv=discr_samp_avg[:,idy]
            conv_avg=convert_brightfld_prot(pre_conv,discr_bkgrd, param_opt)/500.
            val_avgbiom[k][str_f[j]+f"_Rep{k+1}_{6+idy}hour"]=pd.Series(profile_tlapse.nan_adding(rpos_hold[ind_radmax], conv_avg))

    #---Output Formating---#
    df_avgbiom=[]
    for u in range(len(val_avgbiom)):
        g1=pd.DataFrame(val_avgbiom[u])
        df_avgbiom.append(g1)
    power_paramopt["Sample"]=[f"{str_f[j]}_Rep{x+1}" for x in range(len(profile_tlapse.sheets_set[j]))]
    prof_bkgrdsub.append(df_avgbiom)
    res_powerparam.append(power_paramopt)

Minimization complete with experiment disagreement of: 39.653429791162296
Minimization complete with experiment disagreement of: 36.533116959191446
Minimization complete with experiment disagreement of: 0.0011106640652940314


In [ ]:
os.chdir("path/to/Swimplate_Data/Converted-density-profiles/")
for j in range(len(prof_bkgrdsub)):
    with pd.ExcelWriter(f"Density-profile-avg_{str_f[j]}_Tlapse.xlsx") as writer:
        for x in range(len(prof_bkgrdsub[j])):
            prof_bkgrdsub[j][x].to_excel(writer, sheet_name=str_f[j]+f'_Rep{x+1}',index=False)
            
    res_powerparam[j].to_csv(f"Parameter-opt_{str_f[j]}_Tlapse.csv", index=False)

# End-point image data

In [ ]:
os.chdir("path/to/Swimplate_Data/Processed-intensity-profiles/")
str_f=["WT_025ag","WT_028ag","del phzM","del argG","del metH", "del purH"]
# str_f=["del argG","del metH", "del purH"]
endpt_set=[glob.glob(os.getcwd()+f"/SwmPlt-intensity_endpt*{x}*.xlsx")[0] for x in str_f]
profile_endpt=EndPointFile(endpt_set,time_offset=6)
profile_endpt.endpoint_split()
biom_targ=[]
df_biom=pd.read_excel("TotalProtein_BiomTarget.xlsx", sheet_name='Endpoint')
for j in range(len(str_f)):
    biom_targ.append(df_biom[df_biom["Group"]==str_f[j]].loc[:,df_biom.columns[1:]].to_numpy()[0])

Start point of position arrays subtracted 0.1 cm


In [13]:
prof_bkgrdsub=[]
res_powerparam=[]
for j in range(len(profile_endpt.sample_data)):
    r_set=profile_endpt.r_pos[j].to_numpy()
    bkgrd_set=profile_endpt.bkgrd_data[j]
    sample_set=profile_endpt.sample_data[j]
    rad_fulllength=np.zeros(min(len(r_set)//42,38))
    for m in range(len(rad_fulllength)):
        rad_fulllength[m]=np.mean(r_set[42*m:42*(m+1)])
    val_avgbiom={}
    power_paramopt=pd.DataFrame(columns=["Exponent", "Slope"])
    for k in range(sample_set.shape[1]):
        bkgrd_ix=profile_endpt.nan_remove(bkgrd_set[bkgrd_set.columns[k]].to_numpy())
        sample_idx=profile_endpt.nan_remove(sample_set[sample_set.columns[k]].to_numpy())
        r_ix=r_set[:len(sample_idx)]
        
        #---Spatial averaging---#
        discr_r=np.zeros(min(len(r_ix)//42,38)) #floor division towards spatial averaging of ~400 um
        discr_bkgrd=np.zeros(min(len(r_ix)//42,38))
        discr_samp_avg=np.zeros(min(len(r_ix)//42,38))
        for idx in range(len(discr_r)):
            dr=r_ix[42*idx:42*(idx+1)]
            discr_r[idx]=np.mean(dr)
            discr_bkgrd[idx]=radinteg_1D(dr,bkgrd_ix[42*idx:42*(idx+1)])/avgarea_cyl1D(dr[0],dr[-1])
            gry_dr=sample_idx[42*idx:42*(idx+1)]
            avg_gry=radinteg_1D(dr,gry_dr)/avgarea_cyl1D(dr[0],dr[-1])
            discr_samp_avg[idx]=avg_gry

        val_avgbiom["Position (cm)"]=pd.Series(rad_fulllength)
        
        #---Power Law optimization---#
        biom_ix=biom_targ[j][k]
        param_opt, fit_error = optimize_brightfld_prot(discr_samp_avg, discr_bkgrd, discr_r, biom_ix, [-0.6, 60000., -1126.]) # set intercept: -1126.
        print(f"Minimization complete with experiment disagreement of: {fit_error}")
        power_paramopt.loc[len(power_paramopt)]=param_opt
        
        #---Conversion---#
        pre_conv=discr_samp_avg
        conv_avg=convert_brightfld_prot(pre_conv, discr_bkgrd, param_opt)/500.
        val_avgbiom[str_f[j]+f"_Rep{k+1}"]=pd.Series(profile_endpt.nan_adding(rad_fulllength,conv_avg))

    #---Output Formating---#
    power_paramopt["Sample"]=[f"{str_f[j]}_Rep{x+1}" for x in range(sample_set.shape[1])]
    prof_bkgrdsub.append(pd.DataFrame(val_avgbiom))
    res_powerparam.append(power_paramopt)

Minimization complete with experiment disagreement of: 1.3695967027160805e-08
Minimization complete with experiment disagreement of: 1.138118932431098e-09
Minimization complete with experiment disagreement of: 5.6170847528846934e-08
Minimization complete with experiment disagreement of: 3.1333229344454594e-09
Minimization complete with experiment disagreement of: 3.16151727020042e-09
Minimization complete with experiment disagreement of: 4.107482709514443e-08
Minimization complete with experiment disagreement of: 1.1312067726976238e-08
Minimization complete with experiment disagreement of: 9.229324859916233e-09


C:\Users\Sanha Kim\AppData\Local\Temp\ipykernel_52920\1343033672.py:39: RuntimeWarning: invalid value encountered in double_scalars
  pt_sample=b_val*img_profile[j]**a_val


Minimization complete with experiment disagreement of: 3.118657332379371e-08
Minimization complete with experiment disagreement of: 1.1912334230146371e-08
Minimization complete with experiment disagreement of: 9.833570402406622e-09
Minimization complete with experiment disagreement of: 2.56704879575409e-09
Minimization complete with experiment disagreement of: 4.453568180906586e-09
Minimization complete with experiment disagreement of: 2.356500772293657e-09
Minimization complete with experiment disagreement of: 6.327468327071983e-09
Minimization complete with experiment disagreement of: 13.303909522957
Minimization complete with experiment disagreement of: 11.076972858582272
Minimization complete with experiment disagreement of: 13.91560614135507
Minimization complete with experiment disagreement of: 15.143085219014768
Minimization complete with experiment disagreement of: 12.836414507311927
Minimization complete with experiment disagreement of: 18.980847448990005
Minimization complete

In [ ]:
os.chdir("path/to/Swimplate_Data/Converted-density-profiles/")
with pd.ExcelWriter("Density-profile-avg_Endpt.xlsx") as writer:
    for j in range(len(prof_bkgrdsub)):
        prof_bkgrdsub[j].to_excel(writer, sheet_name=str_f[j],index=False)
            
for j in range(len(res_powerparam)):
    res_powerparam[j].to_csv(f"Parameter-opt_{str_f[j]}_Endpt.csv", index=False)